In [1]:
import pymssql
import requests
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

db_settings = {
    "host": "127.0.0.1",
    "user": "user",
    "password": "0000",
    "database": "NCU_database",
    "charset": "utf8"
}

# 儲存台灣50前10的陣列
taiwan50 = []

In [3]:
if __name__ == "__main__":
    find_Taiwan50()
    find_stock("https://isin.twse.com.tw/isin/C_public.jsp?strMode=4", "股票", "特別股")
    find_stock("https://isin.twse.com.tw/isin/C_public.jsp?strMode=2", "股票", "上市認購(售)權證")

The Top 50 stocks:
['2330', '2891', '2883', '2884', '2317', '2890', '2886', '2887', '2303', '2885', '2002', '2892', '5880', '2882', '2880', '2881', '1101', '1303', '1216', '2412', '5876', '1301', '3711', '1326', '3231', '2382', '4938', '2308', '2301', '4904', '2609', '2454', '3045', '3037', '6505', '5871', '2603', '2357', '3034', '2912', '2345', '2379', '2327', '2395', '2207', '3017', '6446', '6669', '3008', '3661']

number of stocks: 50



In [2]:
def find_Taiwan50():
    options = Options()
    options.add_argument("--headless")  # 執行時不顯示瀏覽器
    options.add_argument("--disable-notifications")  # 禁止瀏覽器的彈跳通知
    driver = webdriver.Edge(options=options)
    driver.get("https://www.cmoney.tw/etf/tw/0050/fundholding")
        
    # TODO : 練習2
    time.sleep(5)
    html_list = driver.find_elements(By.CSS_SELECTOR, 'div[class="cm-table"] tbody tr')
    
    taiwan50.clear()
    
    for html in html_list:
        html = html.find_elements(By.CSS_SELECTOR, 'td')
        taiwan50.append(html[0].text)
    
    # 最後兩個不需要
    del taiwan50[-2:]
    print (f"The Top 50 stocks:\n{taiwan50}\n")
    print (f"number of stocks: {len(taiwan50)}\n")
    driver.quit()

def find_stock(url, start, end):
    try:
        conn = pymssql.connect(**db_settings)
        cursor = conn.cursor()
        # TODO : 練習2
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        trs = soup.select("table.h4 > tr")   # 注意，外面一定要是雙引號否則找不到

        # stock 資訊 dict
        stock_dict = {"stock_code": [], "name": [], "type": [], "category": [], "isTaiwan50": []}
        counter = 0 # 啟動開關
        for tr in trs:
            tds = tr.find_all("td")
            row = [td.get_text(strip=True) for td in tds]    # 取出 td 的文字內容
            
            if len(row) == 1 and (row[0] == start or row[0] == end): 
                counter += 1
                continue
            
            # 處理 區間 和 非區間 資料
            if counter < 1:
                continue
            elif counter ==  1:
                stock_code, name = row[0].split("\u3000")
                stock_dict["stock_code"].append(stock_code)
                stock_dict["name"].append(name)
                stock_dict["type"].append(row[3])
                stock_dict["category"].append(row[4])
                stock_dict["isTaiwan50"].append(stock_code in taiwan50)
            else:
                break
        
        # **定義插入語句**
        insert_query = """
            INSERT INTO dbo.stock_info (stock_code, name, type, category, isTaiwan50)
            VALUES (%s, %s, %s, %s, %d)
        """
        for ind, b in enumerate(stock_dict["stock_code"]):
            cursor.execute(insert_query, (stock_dict["stock_code"][ind], stock_dict["name"][ind], 
                                          stock_dict["type"][ind], stock_dict["category"][ind],
                                          stock_dict["isTaiwan50"][ind]))
            conn.commit()

        
    except Exception as e:
       print(e)
       
            
    finally:
        conn.close()